
## What to do after fetching new data

Every time you run the updater you need to rerun three notebooks in order:
```
02_preprocessing.ipynb   → regenerates cves_processed.csv
04_embeddings.ipynb      → generates new embeddings (only for new rows ideally)
05_training.ipynb        → retrains XGBoost on full updated dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import requests, pandas as pd, numpy as np
import time, os, json
from datetime import datetime, timedelta, timezone

BASE    = '/content/drive/MyDrive/CVE_Project'
RAW     = f'{BASE}/raw_data'
API_KEY = "89c76cde-5676-4bb9-94ac-690792a7007b"
HEADERS = {"apiKey": API_KEY}

# ── This file stores the last successful update date automatically ──────
TRACKER_FILE = f'{BASE}/last_updated.json'

def get_last_collected():
    """Read last collected date from tracker file. Falls back to 2023-12-31."""
    if os.path.exists(TRACKER_FILE):
        with open(TRACKER_FILE, 'r') as f:
            data = json.load(f)
            date = datetime.strptime(data['last_collected'], '%Y-%m-%d')
            print(f"Tracker file found. Last collected: {date.strftime('%Y-%m-%d')}")
            return date
    else:
        print("No tracker file found. Using default: 2023-12-31")
        return datetime(2023, 12, 31)

def save_last_collected(date):
    """Save the last collected date to tracker file."""
    with open(TRACKER_FILE, 'w') as f:
        json.dump({'last_collected': date.strftime('%Y-%m-%d')}, f)
    print(f"Tracker updated to: {date.strftime('%Y-%m-%d')}")

def score_to_label(score):
    if score >= 9.0:   return "Critical"
    elif score >= 7.0: return "High"
    elif score >= 4.0: return "Medium"
    else:              return "Low"

def fetch_chunk(start, end):
    BASE_URL  = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    all_items = []
    idx       = 0
    while True:
        url = (f"{BASE_URL}?pubStartDate={start}&pubEndDate={end}"
               f"&startIndex={idx}&resultsPerPage=2000")
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            r.raise_for_status()
            data  = r.json()
            total = data.get("totalResults", 0)
            items = data.get("vulnerabilities", [])
            all_items.extend(items)
            if len(all_items) >= total:
                break
            idx += 2000
            time.sleep(0.7)
        except Exception as e:
            print(f"  Error: {e}")
            break
    return all_items

def parse_items(items, existing_ids):
    rows = []
    for item in items:
        try:
            cve  = item["cve"]
            desc = ""
            for d in cve.get("descriptions", []):
                if d["lang"] == "en":
                    desc = d["value"]
                    break
            if not desc or "** REJECT **" in desc or len(desc.split()) < 10:
                continue
            if cve["id"] in existing_ids:
                continue
            metrics   = cve.get("metrics", {})
            cvss_data = None
            if "cvssMetricV31" in metrics:
                cvss_data = metrics["cvssMetricV31"][0]["cvssData"]
            elif "cvssMetricV30" in metrics:
                cvss_data = metrics["cvssMetricV30"][0]["cvssData"]
            else:
                continue
            score = cvss_data["baseScore"]
            rows.append({
                "cve_id":              cve["id"],
                "description":         desc,
                "cvss_score":          score,
                "cvss_label":          score_to_label(score),
                "attack_vector":       cvss_data.get("attackVector", ""),
                "attack_complexity":   cvss_data.get("attackComplexity", ""),
                "privileges_required": cvss_data.get("privilegesRequired", ""),
                "user_interaction":    cvss_data.get("userInteraction", ""),
                "scope":               cvss_data.get("scope", "")
            })
        except:
            continue
    return rows

# ── MAIN LOGIC ──────────────────────────────────────────────────────────
df             = pd.read_csv(f'{RAW}/cves_raw.csv')
existing_ids   = set(df['cve_id'].tolist())
LAST_COLLECTED = get_last_collected()   # automatically reads from tracker
today          = datetime.now(timezone.utc).replace(tzinfo=None)
gap_days       = (today - LAST_COLLECTED).days

print(f"Today:               {today.strftime('%Y-%m-%d')}")
print(f"Gap:                 {gap_days} days to fill")
print(f"Existing CVEs:       {len(df)}")

if gap_days < 1:
    print("\nDataset is already up to date. Nothing to fetch.")
else:
    # Build 100-day chunks
    chunks  = []
    current = LAST_COLLECTED + timedelta(days=1)
    while current < today:
        chunk_end = min(current + timedelta(days=99), today)
        chunks.append((
            current.strftime('%Y-%m-%dT00:00:00.000'),
            chunk_end.strftime('%Y-%m-%dT23:59:59.999')
        ))
        current = chunk_end + timedelta(days=1)

    print(f"Chunks to fetch:     {len(chunks)}")
    print("\nStarting fetch...\n")

    all_new = []
    for i, (start, end) in enumerate(chunks):
        print(f"Chunk {i+1}/{len(chunks)}: {start[:10]} → {end[:10]}")
        items    = fetch_chunk(start, end)
        new_rows = parse_items(items, existing_ids)
        all_new.extend(new_rows)
        existing_ids.update([r['cve_id'] for r in new_rows])
        print(f"  Added {len(new_rows)} new CVEs | Running total: {len(all_new)}")
        time.sleep(2)

    if all_new:
        new_df   = pd.DataFrame(all_new)
        combined = pd.concat([df, new_df], ignore_index=True)
        combined.to_csv(f'{RAW}/cves_raw.csv', index=False)
        print(f"\nDone.")
        print(f"New CVEs added:  {len(all_new)}")
        print(f"Total now:       {len(combined)}")
        print(f"Saved to {RAW}/cves_raw.csv")
    else:
        print("No new CVEs found in this period.")

    # ── Save today's date as the new last_collected ──────────────────────
    save_last_collected(today)
    print("\nNEXT STEPS:")
    print("1. Re-run 02_preprocessing.ipynb")
    print("2. Run 04_embeddings.ipynb  (appends only new rows)")
    print("3. Re-run 05_training.ipynb")

Mounted at /content/drive
Tracker file found. Last collected: 2026-03-22
Today:               2026-03-22
Gap:                 0 days to fill
Existing CVEs:       200431

Dataset is already up to date. Nothing to fetch.
